In [1003]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
from shapely.geometry import Point, LineString
import os
from gurobipy import quicksum

In [1004]:
batteryCost ={
    "ID"        : [ 'Small',  'medium', 'large'],
    "lower(kWh)": [   1,        101,      1001],
    "upper(kWh)": [   100,       1000,     1e6],
    "Cost/kWh"  : [   520,       420,      400]  
}
# batteryCost ={
#     "ID"        : [ 'Small',  'medium', 'large'],
#     "lower(kWh)": [   1,        101,      1001],
#     "upper(kWh)": [   100,      1000,     1e6],
#     "Cost/kWh"  : [   0.5,       0.4,       0.2]  
# }
df_Batcost =pd.DataFrame(batteryCost)
df5_Batcost= df_Batcost[["ID", "lower(kWh)", "upper(kWh)", "Cost/kWh"]]

In [1005]:
#Demand data
Demand_EV = {
    "UniqueID"       : [4013, 4962, 7323, 9752, 10445, 5494, 11286, 11744, 12863, 13967, 14257, 139, 4, 10829, 12734, 10837, 292, 517, 1338, 816, 849, 956, 1343, 1457],
    "POINT_X"        : [162702.5959, 160806.7076, 163137.5767, 163808.8194, 159304.1657, 160847.2871, 161458.1507, 162023.7627, 158564.5389, 162586.3649, 159885.7536, 160575.478, 161546.1922, 162175.9328, 158131.2171, 160295.2478, 158071.1464, 163881.3241, 161683.7643, 162931.5738, 156259.0608, 161346.8151, 158468.292, 161887.7301],
    "POINT_Y"        : [388343.1643, 384430.2598, 389183.8419, 381057.672, 384714.9847, 380556.4307, 384557.4721, 382560.5136, 381496.1436, 381541.3364, 381695.8029, 387035.874, 385134.076, 382019.7935, 383826.4788, 383584.8842, 387540.4075, 385516.7641, 386983.5747, 381679.7145, 383592.9226, 381981.2913, 388589.9954, 380129.0916],
    "Population"     : [122.7000931, 22.53254502, 4.941334742, 123.374393, 60.70779358, 1.442556602, 15.39575384, 23.28579294, 109.7481401, 11.92150725, 33.00616403, 46.43129092, 26.3520007, 20.53147131, 70.22421862, 58.1324635, 247.9302098, 41.20232361, 53.1136651, 36.63094804, 38.7019138, 12.90018456, 207.4113305, 89.54674616],
    "Solar_Supply_MW": [2.8322, 0.5236, 1.2376, 0.9044, 0.7378, 0.3332, 0.5236, 1.5946, 4.9504, 1.071, 1.428, 1.2614, 2.9512, 0.357, 4.9504, 0.4284, 3.1892, 3.57, 1.9516, 1.071, 3.0464, 0.833, 35.6704, 0.4998] 
}
#Demand_EV  =  pd.read_csv("EV_Demand_2024_eindhoven.csv")
Demand = pd.DataFrame(Demand_EV)

EV_proportion = {
    'Models'               : ['Model Y', 'Model 3', 'EX30', 'XC40', 'Golf', 'ID.4', 'ID.3', 'Aygo X', 'Nero', 'Picanto', 'Captur', 'iX1', 'i4', 'Kona', 'Ioniq 5', '208', 'e-2008', 'Kodiaq'],
    '% Proportion'         : [0.171204758, 0.096139853, 0.097038188, 0.060592722, 0.024255055, 0.01454405, 0.016825822, 0.071220029, 0.098044324, 0.075190672, 0.029618118, 0.034792529, 0.018424859, 0.061976158, 0.006503948, 0.054843375, 0.0365892, 0.03219634],
    'Battery Capacity(kWh)': [60, 57.5, 69, 82, 32, 62, 45, 62, 64.8, 58.3, 9.8, 64.8, 68.7, 64.8, 63, 48.1, 54, 25.7],
    'EV Range(km)'         : [393, 513, 403, 472, 324, 336, 330, 336, 463, 429, 375, 333, 418, 420, 394, 410, 400, 100]
        }
df_EV_proportion = pd.DataFrame(EV_proportion)

# Calculate total battery capacity
Demand['Demand(kWh)'] = 0.0
for index, row in Demand.iterrows():
    total_capacity = 0.0
    population = row['Population']
    for _, ev_row in df_EV_proportion.iterrows():
        proportion = ev_row['% Proportion']
        battery_capacity = ev_row['Battery Capacity(kWh)']
        capacity = population * proportion * battery_capacity
        total_capacity += capacity
    Demand.at[index, 'Demand(kWh)'] = total_capacity

df1_demand = Demand[["UniqueID", "POINT_X", "POINT_Y", "Population", "Demand(kWh)", "Solar_Supply_MW"]].round(2)

In [1006]:
EC = {}# units are kWh/km
for v in range(len(df_EV_proportion)):
    EV_model = df_EV_proportion["Models"].iloc[v]
    capacity = df_EV_proportion["Battery Capacity(kWh)"].iloc[v]
    ev_range = df_EV_proportion["EV Range(km)"].iloc[v]
    EC[EV_model] = capacity / ev_range

In [1007]:
av_EC = sum(EC.values()) / len(EC) # the average energy consumed to move a km.

In [1008]:
#Charging station data.
datacharger = {
    'ID_unique'                             : [3516, 9327, 12309, 3613, 9329, 3518, 12647, 14960, 24046, 8391],
    'DependeVar'                            : [1, 1, 1, 1, 1, 1, 1, 0, 0, 1],
    'POINT_X'                               : [156876.4827, 160013.3214, 158918.6715, 155673.6847, 159494.0311, 156888.2119, 155638.1638, 163714.2396, 157718.7921, 157171.954],
    'POINT_Y'                               : [383419.989, 379813.1895, 380737.7976, 385205.5304, 380139.7906, 384263.4887, 385295.9729, 380840.1225, 387696.0504, 383443.493],
    'LandPrice'                             : [425.980011, 456.3009338, 366.0549927, 719.0772705, 419.8666382, 392.0378418, 717.065979, 824.2540894, 374, 491.4030151],
    'Available Grid intake(Feed-in)(MW)'    : [64, 64, 64, 109, 64, 64, 109, 190, 109, 64],
    'Grid_supply'                           : [75, 77, 77, 111, 77, 77, 111, 207, 111, 77],
    'Possible Location feed-in to Grid(MW)' : [137.9110689, 137.9110689, 137.9110689, 128.9965795, 137.9110689, 137.9110689, 128.9965795, 290.6023465, 128.9965795, 137.9110689],
    'Actual required at location(MW)'       : [110.1706551, 110.1706551, 110.1706551, 121.8170066, 110.1706551, 110.1706551, 121.8170066, 197.6820818, 121.8170066, 110.1706551],
    'Congestion(MW)'                        : [33, 33, 33, 11, 33, 33, 11, -9, 11, 33],
    'Feed_in_congestion(MW)'                : [74, 74, 74, 20, 74, 74, 20, 101, 20, 74]
}
#datacharger = pd.read_csv("Chargers_with_Probability_Greater_than_0.5.csv")
df_chargers = pd.DataFrame(datacharger)
#df_chargers = df_chargers[df_chargers["Probability"] > 0.5]
df2_chargers = df_chargers[['ID_unique', 'LandPrice', 'Grid_supply', 'Actual required at location(MW)', 'Available Grid intake(Feed-in)(MW)']]
#df2_chargers = df2_chargers.reset_index(drop=True)  #reset index after filtering

In [1009]:
print(len(df2_chargers))

10


In [1010]:
# Time and month data
time_data = {
    "Time"                  : ["0:00–7:00", "7:00–11:00", "11:00–17:00", "17:00–24:00"],
    "Daily Yield"           : [0.113, 0.503, 0.444, 0.095],
    "Daily Demand variation": [0.096, 0.278, 0.441, 0.185],
    "time_interval"         : [7, 4, 6, 7],
    "Daily tariff(€/kWh)"   : [0.25, 0.32, 0.22, 0.34] #Euro per kWh
}
df3 = pd.DataFrame(time_data)
df3_time = df3[["Time", "Daily Yield", "time_interval", "Daily Demand variation", "Daily tariff(€/kWh)"]]

In [1011]:
# month_data = {
#     "Month": ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"],
#     "% Monthly yield": [0.035, 0.051, 0.089, 0.118, 0.124, 0.122, 0.12, 0.111, 0.094, 0.064, 0.04, 0.029]
# }
season_data = {
    "season"                 : ["Winter", "Spring", "Summer", "Autumn"],
    "seasonal yield"         : [0.0383, 0.1103, 0.1177, 0.0660],
    "Seasonal tariff (€/kWh)": [0.4,    0.3,    0.25,   0.3],
    "Days in season"         : [90,     92,       92,    91]
}
seasons = pd.DataFrame(season_data)
df4_seasons = seasons[["season", "seasonal yield", "Seasonal tariff (€/kWh)", "Days in season" ]]

In [1012]:
# Fill null values in LandPrice with the mean
Neighborhoods = df1_demand[['POINT_X', 'POINT_Y']].to_numpy()
neighborhood_id = df1_demand["UniqueID"].to_numpy()
Chargers = df_chargers[['POINT_X', 'POINT_Y', "LandPrice", 'Grid_supply']].to_numpy()
charger_id = df_chargers["ID_unique"].to_numpy()

In [1013]:
# Validate input data
if any(Neighborhoods[:, 1] < 0):
    print("Warning: Negative population values detected")
if any(Chargers[:, 1] <= 0):
    print("Warning: Non-positive land costs detected")

In [1014]:
# Road network data- Load shapefile
shapefile_path = "C:/Users/Katon002/PycharmProjects/Optimization/EindhovenSpeedNetwork.shp"
if not os.path.exists(shapefile_path):
    raise FileNotFoundError(f"Shapefile not found at {shapefile_path}")
gdf = gpd.read_file(shapefile_path)
if gdf.crs.to_epsg() != 28992:
    gdf = gdf.to_crs(epsg=28992)

In [1015]:
# CONFIG
DEFAULT_SPEED = 50.0
MIN_SPEED = 15.0
SNAP_TOLERANCE = None   # set to e.g. 0.5 (meters) to snap close coords, or None to disable

# 0) Ensure CRS (must be done earlier)
assert gdf.crs and gdf.crs.to_epsg() == 28992, "gdf must be EPSG:28992 (projected in meters)"

# 1) Preprocess speed column once
gdf["MAXSHD"] = pd.to_numeric(gdf["MAXSHD"], errors="coerce")
gdf["MAXSHD"] = gdf["MAXSHD"].fillna(DEFAULT_SPEED).clip(lower=MIN_SPEED)

# helper: optional snapping function to merge near-equal vertices (disable by setting SNAP_TOLERANCE=None)
def _snap(coord, tol):
    if tol is None:
        return (coord[0], coord[1])
    return (round(coord[0] / tol) * tol, round(coord[1] / tol) * tol)

# 2) Build graph
G = nx.Graph()
coord_to_id = {}
node_id = 0

for _, row in gdf.iterrows():
    geom = row.geometry
    speed_km_h = float(row["MAXSHD"])  # safe now

    lines = list(geom.geoms) if geom.geom_type == "MultiLineString" else [geom]
    for line in lines:
        coords = list(line.coords)
        for k in range(len(coords) - 1):
            s = _snap(coords[k], SNAP_TOLERANCE)
            e = _snap(coords[k+1], SNAP_TOLERANCE)

            if s not in coord_to_id:
                coord_to_id[s] = node_id
                G.add_node(node_id, geometry=Point(s), x=s[0], y=s[1])
                node_id += 1
            if e not in coord_to_id:
                coord_to_id[e] = node_id
                G.add_node(node_id, geometry=Point(e), x=e[0], y=e[1])
                node_id += 1

            u = coord_to_id[s]
            v = coord_to_id[e]

            dx = e[0] - s[0]
            dy = e[1] - s[1]
            distance_km = np.hypot(dx, dy) / 1000.0
            travel_time_hr = distance_km / speed_km_h

            if distance_km > 0 and travel_time_hr > 0:
                G.add_edge(u, v,
                           distance_km=distance_km,
                           travel_time_hr=travel_time_hr,
                           speed_km_h=speed_km_h,
                           weight=travel_time_hr)
            else:
                # very unlikely now; log and continue
                print(f"Warning: invalid edge {u}-{v} (d={distance_km}, t={travel_time_hr})")

G.graph['crs'] = 'EPSG:28992'

In [1016]:
# Check graph connectivity
if not nx.is_connected(G):
    print("Step 5: Warning: Graph is not fully connected. Using largest connected component")
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()

Step 5: Warning: Graph is not fully connected. Using largest connected component


In [1017]:
# Create GeoDataFrames
gdf_neighborhoods = gpd.GeoDataFrame(df1_demand, geometry=gpd.points_from_xy(df1_demand['POINT_X'], df1_demand['POINT_Y']), crs='EPSG:28992')
gdf_chargers = gpd.GeoDataFrame(df_chargers, geometry=gpd.points_from_xy(df_chargers['POINT_X'], df_chargers['POINT_Y']), crs='EPSG:28992')

In [1018]:
# Find nearest nodes
def find_nearest_node(point, graph):
    try:
        return ox.distance.nearest_nodes(graph, X=point.x, Y=point.y)
    except ValueError:
        print(f"Warning: No nearest node found for {point}")
        return None
neighborhood_nodes = [find_nearest_node(pt, G) for pt in gdf_neighborhoods.geometry]
charger_nodes = [find_nearest_node(pt, G) for pt in gdf_chargers.geometry]

In [1019]:
# # Filter valid nodes
valid_neighborhood_idx = [i for i, n in enumerate(neighborhood_nodes) if n is not None]
valid_charger_idx = [j for j, c in enumerate(charger_nodes) if c is not None]
neighborhood_nodes = [n for n in neighborhood_nodes if n is not None]
charger_nodes = [c for c in charger_nodes if c is not None]

Neighborhoods = df1_demand.iloc[valid_neighborhood_idx].to_numpy()
neighborhood_id = df1_demand["UniqueID"].iloc[valid_neighborhood_idx].to_numpy()
Chargers = df_chargers.iloc[valid_charger_idx].to_numpy()
charger_id = df_chargers["ID_unique"].iloc[valid_charger_idx].to_numpy()
neighborhood_nodes = []
removed_neighborhoods = []

#checking for removed neighbnourhoods
for i, pt in enumerate(gdf_neighborhoods.geometry):
    node = find_nearest_node(pt, G)
    if node is None:
        removed_neighborhoods.append(i)  # index of the neighborhood removed
    neighborhood_nodes.append(node)
print(removed_neighborhoods)

[]


In [1020]:
# --- Step 1: Initialize arrays ---
num_neighborhoods = len(neighborhood_nodes)
num_chargers = len(charger_nodes)

travel_distances = np.full((num_neighborhoods, num_chargers), np.inf, dtype=np.float32)
travel_times = np.full((num_neighborhoods, num_chargers), np.inf, dtype=np.float32)

# --- Step 2: Efficient computation using single-source Dijkstra ---
for i, n_node in enumerate(neighborhood_nodes):
    # Compute travel times and distances from this neighborhood to all reachable nodes
    time_lengths = nx.single_source_dijkstra_path_length(G, n_node, weight="travel_time_hr")
    dist_lengths = nx.single_source_dijkstra_path_length(G, n_node, weight="distance_km")

    for j, c_node in enumerate(charger_nodes):
        # If charger is reachable from this neighborhood
        if c_node in time_lengths:
            distance = dist_lengths[c_node]
            # Optional max distance filter (keep only reachable within 1.5×4.4 km)
            # travel_times[i, j] = time_lengths[c_node]
            # travel_distances[i, j] = distance
            if distance <= 4.4 * 1.5:
                travel_distances[i, j] = distance
                travel_times[i, j] = time_lengths[c_node]
            # else remains np.inf

In [1021]:
# # Filter neighborhoods with valid paths
# valid_neighborhood_indices = [i for i in range(num_neighborhoods) 
#                               if any(travel_times[i, j] < 1e10 for j in range(num_chargers))]

# if not valid_neighborhood_indices:
#     raise ValueError("No neighborhoods have valid paths to any charger")

# # Apply filtering consistently
# Neighborhoods = Neighborhoods[valid_neighborhood_indices]
# neighborhood_id = neighborhood_id[valid_neighborhood_indices]
# travel_times = travel_times[valid_neighborhood_indices, :]      # <--- use : to keep all columns
# travel_distances = travel_distances[valid_neighborhood_indices, :]


# # Filter neighborhoods with valid paths
# valid_neighborhood_indices = [i for i in range(num_neighborhoods) if any(travel_times[i, j] < 1e10 for j in range(num_chargers))]
# if not valid_neighborhood_indices:
#     raise ValueError("No neighborhoods have valid paths to any charger")
# Neighborhoods = Neighborhoods[valid_neighborhood_indices]
# neighborhood_id = neighborhood_id[valid_neighborhood_indices]
# travel_times = travel_times[valid_neighborhood_indices]

In [1022]:
# print(travel_distances)
# # print(travel_times.iloc[0,0])
#print(travel_times)

In [1023]:
#travel_times.to_csv("travel_times_real.csv", index=False)

In [1024]:
#travel_times.to_csv("travel_times.csv", index=False)
#travel_times = pd.read_csv("travel_times.csv")
#travel_times.to_csv("travel_times_toy.csv", index=False)
#travel_times = pd.read_csv("travel_times_real.csv")

In [1025]:
# start declarations
# create sets
J = list()  # set of potential charger locations
I = list()  # set of demand locations
T = list()  # Set of solar yield times of the day
M = list()  # seasons
B = list()  # set of indices in cost ranges data

In [1026]:
# declare parameters
ccp   = {}  # capacity of charger j at month m and time t
dem   = {}  # demand at location i in month m in time t
gc    = {}  # Grid capacity available for EV consumption
lc    = {}  # Land cost at EVCS location
tt    = {}  # Travel time kWh
bess  = {}
annualised_bcost = {}
annualised_lc = {}
w={}

In [1027]:
# declare decision variables
F   = {}    # objective function values
X   = {}    # amount of energy from charger j to demand location i in month m and time of the day t
S   = {}    # amount of energy from demand location i to station j in month m and time t
G   = {}    # Amount of surplus energy assigned to the charging station from the grid
Z   = {}    # Amount of energy from a charging station supplied to the grid.
Y   = {}    # binary... open new charger
BatType = {}    # Binary variable for battery type
INV = {}    # Inventory is the amount of energy stored at location j in time.

In [1028]:
# declare constraints/equations
E_fob = {}  # set of objective functions
T_fob = {}  # set of objectives for travel time
B_fob = {}  # set of objectives for Battery storage
EC_fob ={}  # Energy cost from the grid.
E_dem = {}  # demand constraints
E_ccp = {}  # charger capacity
E_cbl = {}  # charger energy balance
E_gc  = {}  # Energy constraint from the grid
E_open= {}  # Energy constraint for opening the charger.
E_s   = {}  # Energy constraint for Supply to charger j
E_z   = {}  # Energy constraint for energy leaving the charger location to the grid
E_g   = {}  # energy constraint for energy assigned to charger j from grid.
E_x   = {}  # energy from charger to meet demand
E_INV = {}  # Available energy that can be stored. 
c_grid={}   # Energy consumption for the daily and seasonal tariffs.

In [1029]:
# demand locations
for i in range(len(neighborhood_id)):
    I.append(i)

In [1030]:
# charger locations
for j in range(len(charger_id)):
    J.append(j)

In [1031]:
#  times of the day
for t in range(len(df3_time["Time"])):
    T.append(t)

In [1032]:
# seasons of the year from 1 to 12
for m in range(len(df4_seasons["season"])):
    M.append(m)

In [1033]:
for b in range(len(df5_Batcost["ID"])):
    B.append(b)

In [1034]:
# initialization of demand# Demand calacualted
for i in I:
    for m in M:
        for t in T:
            Value = ((df1_demand.loc[i, 'Demand(kWh)']/1000) *  df3_time.loc[t, "Daily Demand variation"]) - ((((df1_demand.loc[i, "Solar_Supply_MW"])) * df3_time.loc[t, "time_interval"]) *   #convert to mwh. 
                (df3_time.loc[t, "Daily Yield"] * df4_seasons.loc[m, "seasonal yield"]))
            dem[i, m, t] = 0 if Value < 0 else Value         

In [1035]:
# # initialization of travel time tt calculated
# Pre-allocate tt with inf
tt = np.full((num_neighborhoods, num_chargers), np.inf, dtype=np.float32) # sets the numpy array with Inf initially then the next line populates this with values, and inf division remains inf. Inf implies impossible or avoid at all cost

for i in I:
    for j in J:
        dist = travel_distances[i, j]
        time_val = travel_times[i, j]
        if dist > 0 and not np.isinf(dist) and not np.isinf(time_val):
            tt[i, j] = time_val / (av_EC * dist)
        # Invalid: stays inf

#for j in J:
#   tt[i, j] = travel_times[i,j]/(av_EC * travel_distances[i, j]) #units for tt here are h/kWh. But consider the units in other places to be the same  #tt[i,j,m,t] travel based 0n the time, month but also on the energy in Kilowat transported per unit distance# where RD is is the route distance. and EC is the energy consumotion per travel time unit to reach the stations

In [1036]:
#initialization of gc_grid energy consumption
for j in J:
    for t in T:
        gc[j,t] = df2_chargers.loc[j, 'Grid_supply'] * df3_time.loc[t, "time_interval"] #Keep in mwh not kWh for scalling peurposes otherwisde * 1000 # Convert energy from mWh to kWh for 3hour time interval.(Maybe it should be three-hour intervals)

In [1037]:
# for j in J:
#     lc[j] = df2_chargers.loc[j, "LandPrice"]

In [1038]:
# initialization of charger capacity
for j in J:
    for t in T:
        #ccp[j,t] = (22/1000) * df3_time.loc[t, "time_interval"] #convert to mW for charger capacity#charger capacity is for the 22kw type 2 charger recommended in the EU. This capacity is multiplied by a 3-hour interval to convert i to Energy.
        ccp[j,t]=6e7  ###Change this to the equation above when we run with the total chargers.

In [1039]:
#initialization of cost per kWh
for j in J:
   for b in B:
       #bcost[j,b] = df5_Batcost.loc[b, "Cost/kWh"]
       bess[j,b] = (df5_Batcost.loc[b, "upper(kWh)"])/1000 #convert to mWh for scalling purposes

In [1040]:
#Annualised cost initialisation
def CRF(r, N):
    return r / (1 - (1 + r) ** (-N))

def pv_battery_unit_cost(c_per_kWh, r, N_bat, T):
    n_replacements = (T-1) // N_bat + 1
    pv = sum(c_per_kWh / (1 + r) ** (k * N_bat) for k in range(n_replacements))
    return pv

r = 0.04           # discount rate
Tp = 30            # project horizon
N_bat = 10         # Number of operational years of the battery
CRF_T = CRF(r, Tp) # calling CRF

In [1041]:
# Adjusted battery cost dictionary
for j in J:
    for b in B:
        raw_cost_per_MWh = df5_Batcost.loc[b, "Cost/kWh"] # convert Euros to eurosK and kWh to mWh in this case value is the same in ek/mwh with e/kwh
        # PV of battery over horizon
        pv_unit_cost = pv_battery_unit_cost(raw_cost_per_MWh, r, N_bat, Tp)
        # Annualised per kWh-year
        annualised_bcost[j,b] = (pv_unit_cost * CRF_T)

In [1042]:
# Adjusted land cost dictionary
for j in J:
    raw_land_cost = df2_chargers.loc[j, "LandPrice"]/1000 # land price converted to kEuros
    pv_land = raw_land_cost  # one-shot, no replacement
    annualised_lc[j] = (pv_land * CRF_T)

In [1043]:
for m in M:
    for t in T:
        c_grid[m,t]= (df4_seasons.loc[m, "Seasonal tariff (€/kWh)"] * df3_time.loc[t, "Daily tariff(€/kWh)"]) #value remains the same because e/kwh is same as ek/mwh

In [1044]:
# the optimization model starts here
md = gp.Model("EVCS_optmization")

In [1045]:
# --- Numeric stability & scaling ---
md.setParam("NumericFocus", 3)
md.setParam("ScaleFlag", 2)
md.setParam("FeasibilityTol", 1e-5)
md.setParam("IntFeasTol", 1e-5)

# --- Memory & parallelism ---
md.setParam("SoftMemLimit", 32)   # GB, adjust based on system
md.setParam("MemLimit", 64000)    # MB; increase from 8000
md.setParam("Threads", 4)         # cautious but uses CPU better
md.setParam("NodefileStart", 0.10)
md.setParam("NodefileDir", "C:/Temp")

# --- Presolve, heuristics, and strategy ---
md.setParam("Presolve", 2)
md.setParam("PreSparsify", 1)
md.setParam("Cuts", 2)
md.setParam("Heuristics", 0.10)
md.setParam("MIPFocus", 1)
md.setParam("MIPGap", 0.01)
md.setParam("ImproveStartTime", 60)

# --- Root relaxation ---
md.setParam("Method", 2)      # barrier
md.setParam("Crossover", 0)

# --- Time & logging ---
md.setParam("TimeLimit", 3600)
md.setParam("OutputFlag", 1)
md.setParam("DisplayInterval", 10)
md.setParam("LogFile", "EVCS_model_run.log")

Set parameter NumericFocus to value 3
Set parameter ScaleFlag to value 2
Set parameter FeasibilityTol to value 1e-05
Set parameter IntFeasTol to value 1e-05
Set parameter SoftMemLimit to value 32
Set parameter MemLimit to value 64000
Set parameter Threads to value 4
Set parameter NodefileStart to value 0.1
Set parameter NodefileDir to value "C:/Temp"
Set parameter Presolve to value 2
Set parameter PreSparsify to value 1
Set parameter Cuts to value 2
Set parameter Heuristics to value 0.1
Set parameter MIPFocus to value 1
Set parameter MIPGap to value 0.01
Set parameter ImproveStartTime to value 60
Set parameter Method to value 2
Set parameter Crossover to value 0
Set parameter TimeLimit to value 3600
Set parameter OutputFlag to value 1
Set parameter DisplayInterval to value 10
Set parameter LogFile to value "EVCS_model_run.log"


DECISSION VARIABLES X, S AND Z, G

In [1046]:
# The optimization model starts here for j in J:
#1 Decision variables X energy from fgrom j to i to meet demand and S enegy supply to i from j
# for j in J:
#     for i in I:
#         if travel_distances[i, j] <= (4.4 * 1.5):
#             for m in M:
#                 for t in T:
#                     X[j, i, m, t] = md.addVar(name="X({},{},{},{})".format(j, i, m, t), vtype=GRB.CONTINUOUS)
#                     S[i, j, m, t] = md.addVar(name="S({},{},{},{})".format(i, j, m, t), vtype=GRB.CONTINUOUS)

# Step 1: define the validity function
# --- Create only valid variables ---
for j in J:
    for i in I:
        for m in M:
            for t in T:
                if travel_distances[i, j] <= 4.4 * 1.5:  # validity check
                    X[j, i, m, t] = md.addVar(name=f"X({j},{i},{m},{t})", vtype=GRB.CONTINUOUS)
                    S[i, j, m, t] = md.addVar(name=f"S({i},{j},{m},{t})", vtype=GRB.CONTINUOUS)

# --- Optional: precompute validity sets for summation convenience ---
valid_X = list(X.keys())
valid_S = list(S.keys())



In [1047]:
# --- Step 1: validity functions (examples; adjust logic as needed) ---
def is_valid_Z(j, m, t):
    # Example: only define Z if charger j is potentially open in season m
    return True  # or your logical condition here

def is_valid_G(j, m, t):
    return True  # e.g., only when j has grid access in m,t

#2 Decision variable Z energy flow to grid from chargers j and G grid energy assigned to chanrger j.
valid_Z   = [(j, m, t) for j in J for m in M for t in T if is_valid_Z(j, m, t)]
valid_G   = [(j, m, t) for j in J for m in M for t in T if is_valid_G(j, m, t)]
# Z variables
for j, m, t in valid_Z:
    Z[j, m, t] = md.addVar(name=f"Z({j},{m},{t})", vtype=GRB.CONTINUOUS)
    
# G variables
for j, m, t in valid_G:
    G[j, m, t] = md.addVar(name=f"G({j},{m},{t})", vtype=GRB.CONTINUOUS)
    
# for j in J:
#     for m in M:
#         for t in T:
#             Z[j, m, t] = md.addVar(name="Z({},{},{})".format(j, m, t), vtype=GRB.CONTINUOUS)
#             G[j, m, t] = md.addVar(name="G({},{},{})".format(j, m, t), vtype=GRB.CONTINUOUS)

In [1048]:
#3 Decision variables for I inventory what is possible to be stored
def is_valid_INV(j, m, t):
    return True  # e.g., only if j supports storage in m,t

valid_INV = [(j,m,t)for j in J for m in M for t in T if is_valid_INV(j,m,t)]

# INV variables
for j, m, t in valid_INV:
    INV[j, m, t] = md.addVar(name=f"INV({j},{m},{t})", vtype=GRB.CONTINUOUS)

# for j in J:
#     for m in M:
#         for t in T:
#             INV[j,m,t] = md.addVar(name="INV({},{},{})".format(j, m, t), vtype=GRB.CONTINUOUS)

In [1049]:
#4 Variable for opening a charger at location J
for j in J:
    Y[j] = md.addVar(name="Y({})".format(j), vtype=GRB.BINARY)

In [1050]:
for j in J:
    for b in B:
        BatType[j,b] = md.addVar(name="BatType({},{})".format(j,b), vtype=GRB.BINARY)

THE OBJECTIVE FUNCTION: LAND COST AND TOTAL TRAVEL TIME

In [1051]:
# objective function variables initialized
F["LC"] =md.addVar(name="F(LC)", vtype=GRB.CONTINUOUS, lb=0)
F["TT"] =md.addVar(name="F(TT)", vtype=GRB.CONTINUOUS, lb=0)
F["BC"] =md.addVar(name="F(BC)", vtype=GRB.CONTINUOUS, lb=0)
F["EC"] =md.addVar(name="F(EC)", vtype=GRB.CONTINUOUS, lb=0)

CONSTRAINTS

In [1052]:
unreachable = [i for i in range(num_neighborhoods)
               if all(np.isinf(travel_distances[i, j]) for j in range(num_chargers))]
for i in unreachable:
    for m in M:
        for t in T:
            dem[i,m,t] = 0

In [1053]:
# #energy Balance X, S, and Dem
for i in I:
    for m in M:
        for t in T:
            E_dem[i, m, t] = md.addConstr(gp.quicksum(X[j, i, m, t] for j in J if (j, i, m, t) in X) - gp.quicksum(S[i, j, m, t] for j in J if (i, j, m, t) in S) == dem[i, m, t],
                name=f"E_dem({i},{m},{t})")            

# for i in I:
#     for m in M:
#         for t in T:
#             E_dem[i,m,t] = md.addConstr(gp.quicksum((X[j, i, m, t] - S[i, j, m, t]) for j in J if (j, i, m, t) in X) == dem[i,m,t], "E_dem({},{},{})".format(i,m,t))
# #energy Balance X, S, and Dem
# for i in I:
#     for m in M:
#         for t in T:
#             if dem[i, m, t] == 0:
#                 continue  # skip zero-demand entries entirely (no constraint needed)
#                 E_dem[i,m,t] = md.addConstr(gp.quicksum((X[j, i, m, t] - S[i, j, m, t]) for j in J if (j, i, m, t) in X) == dem[i,m,t], "E_dem({},{},{})".format(i,m,t))               
# for i in I:
#     for m in M:
#         for t in T:
            #E_dem[i, m, t] = md.addConstr(gp.quicksum(X[j, i, m, t] for j in J if (j, i, m, t) in X) - gp.quicksum(S[i, j, m, t] for j in J if (i, j, m, t) in S) == dem[i, m, t],name=f"E_dem({i},{m},{t})")
#           E_dem[i, m, t] = md.addConstr(gp.quicksum(X[j, i, m, t] for j in J if (j, i, m, t) in X) - gp.quicksum(S[i, j, m, t] for j in J if (i, j, m, t) in S) == dem[i, m, t],name=f"E_dem({i},{m},{t})")

In [1054]:
# #add X constraint with Y[j]
# for j in J:
#     for m in M:
#         for t in T:
#             E_x[j, m, t] = md.addConstr(gp.quicksum(X[j, i, m, t] for i in I) <= ccp[j,t] * Y[j], name="E_x({},{},{})".format(j, m, t))
# for j in J:
#     for m in M:
#         for t in T:
#             E_x[j, m, t] = md.addConstr(gp.quicksum(X[j, i, m, t] for i in I if (j, i, m, t) in X) <= ccp[j, t] * Y[j], name=f"E_x({j},{m},{t})")
for j in J:
    for m in M:
        for t in T:
            E_x[j, m, t] = md.addConstr(gp.quicksum(X[j, i, m, t] for i in I if (j, i, m, t) in valid_X)<= ccp[j, t] * Y[j], name=f"E_x({j},{m},{t})")


In [1055]:
# # initialization of the big M for the charging station
# Precompute BigMs for each charger j
# BigMs = {j: bess[j,b]*1.1 for j in J}  
# for i in I:
#     for m in M:
#         for t in T:
#             # Only sum over valid j for this i
#             valid_j = [j for j in J if (i,j) in valid_pairs]
#             if valid_j:
#                 E_s[i, m, t] = md.addConstr(gp.quicksum(S[i,j,m,t] for j in valid_j) <= gp.quicksum(BigMs[j]*Y[j] for j in valid_j), "E_sl({},{},{})".format(i, m, t))

# for i in I:
#     for m in M:
#         for t in T:
#             E_s[i, m, t] = md.addConstr(gp.quicksum(S[i, j, m, t] for j in J if (i, j, m, t) in S) <= gp.quicksum(bess[j, b] * 1.1 * Y[j] for j in J for b in B), name=f"E_sl({i},{m},{t})")
for i in I:
    for m in M:
        for t in T:
            E_s[i, m, t] = md.addConstr(gp.quicksum(S[i, j, m, t] for j in J if (i, j, m, t) in valid_S) <= gp.quicksum(bess[j, b] * 1.1 * Y[j] for j in J for b in B), name=f"E_sl({i},{m},{t})")

#Question: Should there be a sum for both the left and right sides, or should it be summed

In [1056]:
#Big M for the energy consumption from the grid
# Define BigMg per j,t (aggregate over b; assume B is the set of batteries)
BigMg = {}
for j in J:
    for t in T:
        BigMg[j, t] = ccp[j, t] + sum(bess[j, b] for b in B)  # Sum storage if multiple b

# Add one constraint per m,t: total grid draw <= 1.1 * total opened capacities
for m in M:
    for t in T:
        E_g[m, t] = md.addConstr(gp.quicksum(G[j, m, t] for j in J if (j, m, t) in valid_G) <= gp.quicksum(BigMg[j, t] * 1.1 * Y[j] for j in J if any((j,m_,t) in valid_G for m_ in M)),
            name=f"E_g({m},{t})")

# for m in M:
#     for t in T:
#         E_g[m, t] = md.addConstr(
#             gp.quicksum(G[j, m, t] for j in J) <= gp.quicksum(BigMg[j, t] * 1.1 * Y[j] for j in J),  # RHS: scaled per-j capacities
#             name=f"E_g({m},{t})")

# BigMg = ccp[j,t] + bess[j,b]
# for j in J:
#     for m in M:
#         for t in T:
#             E_g[j,m,t] = md.addConstr(gp.quicksum(G[j,m,t] for j in J) <= gp.quicksum(BigMg * 1.1 * Y[j] for j in J), "E_g({},{},{})".format(j,m,t)) #should bigM be summed or not

In [1057]:
#Balancing energy flow constraint # Changed gc[m,t] to gc[j]
for j in J:
    for m in M:
        for t in T:
            if (j,m,t) in valid_Z and (j,m,t) in valid_G:
                E_gc[j,m,t] = md.addConstr(Z[j,m,t] - G[j,m,t] <= gc[j,t], name=f"E_gc({j},{m},{t})")
# for j in J:
#     for m in M:
#         for t in T:
#             E_gc[j,m,t] = md.addConstr(Z[j,m,t] - G[j,m,t] <= gc[j,t],name=f"E_gc({j},{m},{t})")
# for m in M:
#     for t in T:
#         E_gc[m,t] = md.addConstr(gp.quicksum(Z[j,m,t] - G[j,m,t] for j in J) <= gc[j,t], "E_gc({},{})".format(m,t))
        #E_gc[m, t] = md.addConstr(sum(Z[j, m, t] - G[j, m, t] for j in J) <= sum(gc[j, t] for j in J), "E_gc({},{})".format(m,t))   #Su gc or not sum

In [1058]:
#Initialization of the big M for the charger to the Grid energy supply: constraints for supply to the grid
for j, m, t in valid_Z:
    md.addConstr(Z[j, m, t] <= gp.quicksum(bess[j, b] * Y[j] for b in B if (j,b) in bess), name=f"Z_max({j},{m},{t})")

# for j in J:
#     for m in M:
#         for t in T:
#             md.addConstr(Z[j,m,t] <= bess[j,b] * Y[j], name=f"Z_max({j},{m},{t})")

# BigMz = bess[j,b] + dem[i,m,t]
# for m in M:
#     for t in T:
#         E_z[j,m,t] = md.addConstr(gp.quicksum(Z[j,m,t] for j in J) <= gp.quicksum(BigMz * 1.1 * Y[j] for j in J), "E_z({},{},{})".format(j,m,t))

In [1059]:
# # #Inventory constraints
# for j in J:
#     for m in M:
#         # t = 0
#         E_INV[j, m, 0] = md.addConstr(
#             INV[j, m, 0] == (
#                 gp.quicksum(S[i, j, m, 0] for i in I if (i, j, m, 0) in S) + G[j, m, 0] - Z[j, m, 0]- gp.quicksum(X[j, i, m, 0] for i in I if (j, i, m, 0) in X)), name=f"E_INV({j},{m},0)")
#         # t > 0
#         for t in T:
#             if t != 0:
#                 E_INV[j, m, t] = md.addConstr(INV[j, m, t] == (INV[j, m, t - 1] + gp.quicksum(S[i, j, m, t] for i in I if (i, j, m, t) in S) + G[j, m, t]- Z[j, m, t] - gp.quicksum(X[j, i, m, t] for i in I if (j, i, m, t) in X)), name=f"E_INV({j},{m},{t})")

# Assuming T is a list of time periods, e.g., T = [0, 1, 2, 3]
# T_last = max(T)  # Last period index

# for j in J:
#     for m in M:
#         # -------------------------
#         # t = 0: use last period as previous inventory (cyclic)
#         # -------------------------
#         E_INV[j, m, 0] = md.addConstr(INV[j, m, 0] == ( INV[j, m, T_last]  # cyclic link
#                 + gp.quicksum(S[i, j, m, 0] for i in I if (i, j, m, 0) in S) + G[j, m, 0]  - gp.quicksum(X[j, i, m, 0] for i in I if (j, i, m, 0) in X)),
#             name=f"E_INV({j},{m},0)")
#         # -------------------------
#         # t > 0: normal inventory evolution
#         # -------------------------
#         for t in T:
#             if t != 0:
#                 E_INV[j, m, t] = md.addConstr(
#                     INV[j, m, t] == (INV[j, m, t - 1] + gp.quicksum(S[i, j, m, t] for i in I if (i, j, m, t) in S) + G[j, m, t]- Z[j, m, t]- gp.quicksum(X[j, i, m, t] for i in I if (j, i, m, t) in X) ), name=f"E_INV({j},{m},{t})"
#                 )

T_last = max(T)  # Last period index

for j, m, t in valid_INV:
    # -------------------------
    # t = 0: use last period as previous inventory (cyclic)
    # -------------------------
    if t == 0:
        E_INV[j, m, t] = md.addConstr(
            INV[j, m, t] == (
                INV[j, m, T_last] if (j, m, T_last) in valid_INV else 0  # cyclic link, 0 if not valid
                + gp.quicksum(S[i, j, m, t] for i in I if (i, j, m, t) in valid_S) + (G[j, m, t] if (j, m, t) in valid_G else 0)- gp.quicksum(X[j, i, m, t] for i in I if (j, i, m, t) in valid_X)
            ),name=f"E_INV({j},{m},{t})" )
    # -------------------------
    # t > 0: normal inventory evolution
    # -------------------------
    else:
        E_INV[j, m, t] = md.addConstr(
            INV[j, m, t] == (INV[j, m, t-1] if (j, m, t-1) in valid_INV else 0+ gp.quicksum(S[i, j, m, t] for i in I if (i, j, m, t) in valid_S)+ (G[j, m, t] if (j, m, t) in valid_G else 0)
                - (Z[j, m, t] if (j, m, t) in valid_Z else 0) - gp.quicksum(X[j, i, m, t] for i in I if (j, i, m, t) in valid_X)),name=f"E_INV({j},{m},{t})")


In [1060]:
for j,m,t in valid_INV:
    md.addConstr(INV[j,m,t] <= gp.quicksum(BatType[j,b] * bess[j,b] for b in B if (j,b) in bess), name=f"Cap_def({j},{m},{t})")

# for j in J:
#     for m in M:
#         for t in T:
#             md.addConstr(INV[j, m, t] <= gp.quicksum(BatType[j, b] * bess[j, b] for b in B), name="Cap_def({},{},{})".format(j, m, t))

In [1061]:
for j in J:
    #md.addConstr(gp.quicksum(BatType[j,b] for b in B) == Y[j], name="SelectOneBattery({})".format(j)) 
    md.addConstr(gp.quicksum(BatType[j,b] for b in B) <= 1, name="SelectOneBattery({})".format(j)) # Alternatively, relax the == 1 constraint to <= 1 to allow no battery selection if infeasible: but then the model easily defaults to no battery

In [1062]:

for m in M:
    for t in T:
        w[m,t]= df4_seasons.loc[m, "Days in season"] * df3_time.loc[t, "time_interval"]

In [1063]:
#Objective for Annual energy consumption cost were c_grid is cost of energy per m and t and w is the number of hours in a year and G energy from grid.
EC_fob["EC"] = md.addConstr(F["EC"] == gp.quicksum(c_grid[m, t] * G[j, m, t] * w[m, t] for (j, m, t) in valid_G), name="E_fob_EC")

# EC_fob["EC"] = md.addConstr(F["EC"] == gp.quicksum(c_grid[m, t] * G[j, m, t] * w[m, t] for j in J for m in M for t in T), name="E_fob_EC")

In [1064]:
# Define the objective function for land costs
E_fob["LC"] = md.addConstr(F["LC"] == gp.quicksum(annualised_lc[j] * Y[j] for j in J), "E_F(LC)")

In [1065]:
# # Define the objective function of total travel time
T_fob["TT"] = md.addConstr(F["TT"] == gp.quicksum((S[i, j, m, t] if (i, j, m, t) in valid_S else 0) + (X[j, i, m, t] if (j, i, m, t) in valid_X else 0) * tt[i, j] for i in I for j in J for m in M for t in T if np.isfinite(tt[i,j])
    ), name="T_fob_TT")

#T_fob["TT"] = md.addConstr(F["TT"] == gp.quicksum(((S[i, j, m, t] if (i, j, m, t) in S else 0) + (X[j, i, m, t] if (j, i, m, t) in X else 0)) * tt[i, j] for i in I for j in J for m in M for t in T if np.isfinite(tt[i,j])), name="T_fob_TT")
#T_fob["TT"] = md.addConstr(F["TT"] == sum((S[i, j, m, t] + X[j, i, m, t]) * tt[i, j] for i in I for j in J for m in M for t in T),"T_fob_TT")
#T_fob["TT"] = md.addConstr(F["TT"] == sum((S[i, j, m, t] + X[j, i, m, t]) * tt[i, j] for i in I for j in J for m in M for t in T), "T_fob_TT") # time per kilowats but removed Y[j]

In [1066]:
# Define objective for battery storage cost
#B_fob["BC"] = md.addConstr(F["BC"] == sum(bess[j,b] * bcost[j,b] * BatType[j,b]  for j in J for b in B), name="B_fob_BC") #original
B_fob["BC"] = md.addConstr(F["BC"] == gp.quicksum(bess[j,b] * annualised_bcost[j,b] * BatType[j,b] for j in J for b in B), name="B_fob_BC")

In [1067]:
# def printSolution(title):
#     print(f"=== Report of model {title} starts here ===\n")

#     if md.status == GRB.OPTIMAL:
#         print("\nOptimization Results:")

#         # Show all objectives, not just md.objVal
#         print("\nObjective Values (multi-objective outcome):")
#         print(f"  Travel Time (TT): {F['TT'].X:.2f}")
#         print(f"  Land Cost   (LC): {F['LC'].X:.2f}")
#         print(f"  BESS Cost   (BC): {F['BC'].X:.2f}")
#         print(f"  Total Cost (LC+BC): {F['LC'].X + F['BC'].X:.2f}")

#         total_chargers = 0
#         total_land_cost = 0.0
#         total_travel_time = 0.0

#         print("\nSelected EVCS Locations:")
#         for j in J:
#             if Y[j].X >= 1e-9:  # station is open
#                 total_chargers += 1
#                 total_land_cost += annualised_lc[j] * Y[j].X
#                 total_travel_time += sum(
#                     tt[i, j] * X[i, j].X for i in I if (i, j) in X
#                 )

#                 print(f"Station {j} open (Land Cost = {annualised_lc[j] * Y[j].X:.2f} Euros)")
#                 print(f"Y({j}) = {round(Y[j].X)}")

#                 # Print decision variables for open chargers only
#                 print(f"-- X variables for Station {j}:")
#                 for key, var in X.items():
#                     if key[1] == j and var.X > 0:
#                         print(f"{var.VarName} = {var.X:.6f}")

#                 print(f"-- S variables for Station {j}:")
#                 for key, var in S.items():
#                     if key[1] == j and var.X > 0:
#                         print(f"{var.VarName} = {var.X:.6f}")

#                 print(f"-- INV variables for Station {j}:")
#                 for key, var in INV.items():
#                     if key[1] == j and var.X > 0:
#                         print(f"{var.VarName} = {var.X:.6f}")

#                 print(f"-- G variables for Station {j}:")
#                 for key, var in G.items():
#                     if key[1] == j and var.X > 0:
#                         print(f"{var.VarName} = {var.X:.6f}")

#                 print(f"-- Z variables for Station {j}:")
#                 for key, var in Z.items():
#                     if key[1] == j and var.X > 0:
#                         print(f"{var.VarName} = {var.X:.6f}")

#         print(f"\nTotal Chargers Opened: {total_chargers}")
#         print(f"Total Land Cost: {F['LC'].X:.2f}")
#         print(f"Total Travel Time: {F['TT'].X:.2f}")
#         print(f"Total BESS Cost: {F['BC'].X:.2f}")

#     elif md.status == GRB.INFEASIBLE:
#         print("Model is infeasible.")
#         md.computeIIS()
#         md.write("model.ilp")
#         print("IIS written to model.ilp")

#     elif md.status == GRB.UNBOUNDED:
#         print("Problem is unbounded.")

#     else:
#         print(f"Solver ended with status {md.status}")

# # First, set the FixedCosts objective
# md.setObjective(F["LC"] + F["BC"], GRB.MINIMIZE)
# md.optimize()

# # Save the optimal value
# if md.status == GRB.OPTIMAL:
#     fixed_val = md.ObjVal
#     print(f"FixedCosts optimal value: {fixed_val:.6f}")
# else:
#     raise ValueError("FixedCosts optimization did not converge")
# # Clear previous objective (multi-objective structures)
# md.setObjective(0)   # removes objective
# md.update()          # ensure the model registers the change

# tolerance = 1e-4  # allow tiny slack
# md.addConstr(F["LC"] + F["BC"] <= fixed_val * (1 + tolerance), "Fix_FC")
# md.setObjective(F["TT"], GRB.MINIMIZE)
# md.optimize()




# # Clear any previous single-objective definition
# md.setObjective(0)

# # 1. Define objectives
# md.setObjectiveN(F["LC"] + F["BC"], index=0, priority=2, weight=1.0, name="FixedCosts")
# md.setObjectiveN(F["TT"], index=1, priority=1, weight=1.0, name="TravelTime")

# # 2 Optimize the model
# md.optimize()

# #  Print the results
# printSolution("EVCS Model Results")

# Example run: minimize fixed costs
#minimization of LC and BC time
# 1️ First stage
# md.setObjective(F["LC"] + F["BC"], GRB.MINIMIZE)
# md.optimize()
# md.addConstr(F["LC"] + F["BC"] <= F["LC"].X + F["BC"].X)
# md.setObjective(F["TT"], GRB.MINIMIZE)
# md.optimize()
# printSolution(" -- results minin  mize TT subject to LC and BCe")

# ##### answer subquestion (a) Make sure to avoid weak 
# #minimize total fixed costs
# md.setObjective(F["TT"], GRB.MINIMIZE)
# md.optimize()
# md.addConstr(F["TT"] <= (F["TT"].X) *(1 + epsilon))
# md.setObjective(F["LC"] + F["BC"], GRB.MINIMIZE)
# md.optimize()
# printSolution("--minimize LC + BC subject to TT--")

In [1068]:
#Checking for invenntory on the right and on the left of the inventory balance
# for j in J:
#      for m in M:
#          for t in T:
#              if t != 0:  # skip t=0 to avoid INV[j,m,t-1] error
#                  print(
#                      "INV[{},{},{}] ({}) = INV[{},{},{}] ({}) + sum(S[i,{},{},{}] for i)({}) + G[{},{},{}]({}) - Z[{},{},{}]({}) - sum(X[{},i,{},{}] for i)({})".format(j, m, t, INV[j, m, t].X,j, m, t-1, INV[j, m, t-1].X,
#                          j, m, t, sum(S[i, j, m, t].X for i in I),
#                          j, m, t, G[j, m, t].X,
#                          j, m, t, Z[j, m, t].X,
#                          j, m, t, sum(X[j, i, m, t].X for i in I)
#                      )
#                  )

In [1069]:
def printSolutionSummary(title):
    print(f"=== Report of model {title} starts here ===\n")

    if md.status == GRB.OPTIMAL:
        print("\nObjective Values:")
        print(f"  Travel Time (TT)  : {F['TT'].X:.2f}")
        print(f"  Land Cost   (LC)  : {F['LC'].X:.2f}")
        print(f"  BESS Cost   (BC)  : {F['BC'].X:.2f}")
        print(f"  Energy cost (EC)  : {F['EC'].X:.2f}")
        print(f"  Total Cost (LC+EC+BC): {F['LC'].X + F['EC'].X + F['BC'].X:.2f}\n")
        

        total_chargers = 0
        total_land_cost = 0.0
        total_travel_time = 0.0

        print("Selected EVCS Locations and Variable Sums:")
        for j in J:
            if Y[j].X >= 1e-9:  # station is open
                total_chargers += 1
                total_land_cost += annualised_lc[j] * Y[j].X
        
                # --- sums per station, using validity sets ---
                sum_X = sum(X[j, i, m, t].X for (j2, i, m, t) in valid_X if j2 == j)
                sum_S = sum(S[i, j, m, t].X for (i, j2, m, t) in valid_S if j2 == j)
                sum_Z = sum(Z[j, m, t].X for (j2, m, t) in valid_Z if j2 == j)
                sum_G = sum(G[j, m, t].X for (j2, m, t) in valid_G if j2 == j)
                sum_INV = sum(INV[j, m, t].X for (j2, m, t) in valid_INV if j2 == j)
                sum_tt = sum(
                    tt[i, j] * X[j, i, m, t].X
                    for (j2, i, m, t) in valid_X if j2 == j and np.isfinite(tt[i,j])
                )
                total_travel_time += sum_tt

                # --- print station summary ---
                print(f"\nStation {j} open:")
                print(f"  Y({j}) = {int(round(Y[j].X))}")
                print(f"  Sum X   = {sum_X:.6f}")
                print(f"  Sum S   = {sum_S:.6f}")
                print(f"  Sum Z   = {sum_Z:.6f}")
                print(f"  Sum G   = {sum_G:.6f}")
                print(f"  Sum INV = {sum_INV:.6f}")
                print(f"  Travel Time (station) = {sum_tt:.6f}")
            

                # Optional: sanity check on zeros
                if abs(sum_S) < 1e-9 and abs(sum_Z) < 1e-9:
                    print("  Note: S and Z are zero (inactive for this station).")

        # --- totals ---
        print(f"\nTotal Chargers Opened: {total_chargers}")
        print(f"Total Land Cost:   {total_land_cost:.6f}€")
        print(f"Total Travel Time: {total_travel_time:.6f}")
        print(f"Total BESS Cost:   {F['BC'].X:.6f}€")
        print(f"Energy Cost :  {F['EC'].X:.6f}€")
    

        # --- global nonzero overview ---
        print("\nSummary of Nonzero Variables:")
        for name, d in [("Y", Y), ("X", X), ("S", S), ("G", G), ("Z", Z), ("INV", INV)]:
            n_nonzero = sum(1 for v in d.values() if abs(v.X) > 1e-9)
            print(f"  {name}: {n_nonzero}/{len(d)} nonzero")

    elif md.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
        md.computeIIS()
        md.write("model.ilp")
        print("IIS written to model.ilp")

    elif md.status == GRB.UNBOUNDED:
        print("Problem is unbounded.")

    else:
        print(f"Solver ended with status {md.status}")

# First, set the FixedCosts objective
md.setObjective(F["LC"] + F["BC"] + F["EC"], GRB.MINIMIZE)
md.optimize()

# # Save the optimal value
# if md.status == GRB.OPTIMAL:
#     fixed_val = md.ObjVal
#     print(f"FixedCosts optimal value: {fixed_val:.6f}")
# else:
#     raise ValueError("FixedCosts optimization did not converge")
# # Clear previous objective (multi-objective structures)
# md.setObjective(0)   # removes objective
# md.update()          # ensure the model registers the change

# tolerance = 1e-4  # allow tiny slack
# md.addConstr(F["LC"] + F["BC"] + F["EC"]<= fixed_val * (1 + tolerance), "Fix_FC")
# md.setObjective(F["TT"], GRB.MINIMIZE)
# md.optimize()
printSolutionSummary("Second Stage (Min TT)")

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: AMD Ryzen Threadripper PRO 5945WX 12-Cores, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 24 logical processors, using up to 4 threads

Non-default parameters:
TimeLimit  3600
MemLimit  64000
SoftMemLimit  32
FeasibilityTol  1e-05
MIPGap  0.01
Method  2
ScaleFlag  2
Crossover  0
Heuristics  0.1
MIPFocus  1
NodefileStart  0.1
Cuts  2
DisplayInterval  10
NumericFocus  3
Presolve  2
PreSparsify  1
Threads  4
ImproveStartTime  60

Optimize a model with 1598 rows, 4108 columns and 16906 nonzeros
Model fingerprint: 0x31196789
Variable types: 4068 continuous, 40 integer (40 binary)
Coefficient statistics:
  Matrix range     [2e-02, 7e+07]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-04, 1e+03]
Presolve removed 892 rows and 2269 columns
Presolve time: 0.01s
Presolved: 706 rows, 1839 columns, 6006 nonzeros
Variable types: 1831 continuous,

In [1070]:
print(f"Estimated RAM use: {md.NumNZs * 8 / 1e9:.2f} GB")

Estimated RAM use: 0.00 GB


In [1071]:
md.printStats()

Statistics for model 'EVCS_optmization':
  Problem type                : MIP
  Linear constraint matrix    : 1598 rows, 4108 columns, 16906 nonzeros
  Variable types              : 4068 continuous, 40 integer (40 binary)
  Matrix range                : [2e-02, 7e+07]
  Objective range             : [1e+00, 1e+00]
  Bounds range                : [1e+00, 1e+00]
  RHS range                   : [3e-04, 1e+03]


In [1072]:
print("Counts:")
print("  |J|,|I|,|M|,|T| =", len(J), len(I), len(M), len(T))
print("  len(valid_X)  =", len(valid_X))
print("  len(valid_S)  =", len(valid_S))
print("  len(valid_Z)  =", len(valid_Z))
print("  len(valid_G)  =", len(valid_G))
print("  len(valid_INV)=", len(valid_INV))
print("  #Y binaries   =", sum(1 for _ in Y))
print("  #BatType bins =", sum(1 for _ in BatType))


Counts:
  |J|,|I|,|M|,|T| = 10 24 4 4
  len(valid_X)  = 1792
  len(valid_S)  = 1792
  len(valid_Z)  = 160
  len(valid_G)  = 160
  len(valid_INV)= 160
  #Y binaries   = 10
  #BatType bins = 30


In [1073]:
# build demand_coverage from valid_X
demand_coverage = {}
for (j,i,m,t) in valid_X:
    demand_coverage.setdefault((i,m,t), []).append(j)

# stats
total_demands = len(I) * len(M) * len(T)
covered_demands = len(demand_coverage)
pct_covered = 100 * covered_demands / total_demands
print(f"Demand tuples total: {total_demands}, covered: {covered_demands} ({pct_covered:.1f}%)")

# list orphan demands (if any)
orphan_demands = [(i,m,t) for i in I for m in M for t in T if (i,m,t) not in demand_coverage]
print("Orphan demands (i,m,t) count:", len(orphan_demands))
if orphan_demands:
    print("Sample orphan demands:", orphan_demands[:20])

Demand tuples total: 384, covered: 352 (91.7%)
Orphan demands (i,m,t) count: 32
Sample orphan demands: [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 0, 3), (0, 1, 0), (0, 1, 1), (0, 1, 2), (0, 1, 3), (0, 2, 0), (0, 2, 1), (0, 2, 2), (0, 2, 3), (0, 3, 0), (0, 3, 1), (0, 3, 2), (0, 3, 3), (2, 0, 0), (2, 0, 1), (2, 0, 2), (2, 0, 3)]


In [1074]:
print("Model counts: Vars:", md.NumVars, "Constrs:", md.NumConstrs, "Nonzeros:", md.NumNZs)
# If you expect a demand constraint per (i,m,t), check:
expected_demand_constraints = len(I)*len(M)*len(T)
print("Expected demand constraints (if one per i,m,t):", expected_demand_constraints)
# You can also manually count constraints with names or patterns if you named them when adding.

Model counts: Vars: 4108 Constrs: 1598 Nonzeros: 16906
Expected demand constraints (if one per i,m,t): 384


In [1075]:
unreachable = [i for i in range(num_neighborhoods)
               if all(np.isinf(travel_distances[i, j]) for j in range(num_chargers))]

print(f"{len(unreachable)} neighborhoods are unreachable under the distance limit.")
print("Example indices:", unreachable[:10])

2 neighborhoods are unreachable under the distance limit.
Example indices: [0, 2]


In [1076]:
# # 1) Are any BatType selected?
# print("BatType nonzero count:",
#       sum(1 for (j,b),v in BatType.items() if abs(v.X) > 1e-9))
# print("Example BatType values (first 20):", [(k,v.X) for k,v in list(BatType.items())[:20]])

# # 2) Are bat capacities strictly zero?
# print("Capacity RHS examples (bess*BatType) for first 10 j:", 
#       [ (j, sum(bess[j,b]*BatType[j,b].X for b in B)) for j in list(J)[:10] ])

# # 3) Show the Big-M or the Z values:
# print("Z extremes:", min(v.X for v in Z.values()), max(v.X for v in Z.values()))
# print("G extremes:", min(v.X for v in G.values()), max(v.X for v in G.values()))

# # 4) Check whether F['BC'] expression contains BatType:
# print("F['BC'] is Var?", isinstance(F['BC'], gp.Var), "value:", F['BC'].X)


In [1077]:
# for v in md.getVars():
#     if abs(v.X) > 0:
#         print(f"{v.VarName} = {v.X}")

In [1078]:
#Printing variables that are available# print("Y values:")
# for j, var in X.items():
#     print(f"{var.VarName} = {var.X}")
    
# print("\n=== Nonzero Decision Variables ===")
# for v in md.getVars():
#     if abs(v.X) >= 1e-6:
#         print(f"{v.VarName} = {v.X:.6f}")


In [1079]:
# # ---------- Diagnostic block: paste after md.optimize() ----------
# tol = 1e-8

# print("Model status:", md.Status, md.Status==GRB.OPTIMAL and "OPTIMAL" or "")
# print("Solution count:", md.SolCount)

# # Quick global nonzero check (fastish)
# nonzero_count = 0
# for v in md.getVars():
#     if abs(v.X) > tol:
#         nonzero_count += 1
# print("Total nonzero decision vars:", nonzero_count)

# # Function to report summary for each dict
# def report_var_dict(name, vdict, sample=10):
#     total = len(vdict)
#     nn = sum(1 for v in vdict.values() if abs(v.X) > tol)
#     print(f"\n{name}: total vars = {total}, nonzero = {nn}")
#     if nn>0:
#         # print up to `sample` nonzeros for inspection
#         printed = 0
#         for k,v in vdict.items():
#             if abs(v.X) > tol:
#                 print(f"  {v.VarName} ({k}) = {v.X:.6f}")
#                 printed += 1
#                 if printed >= sample:
#                     break
#     else:
#         print("  all zero (or below tol)")

# # Run reports for your core groups
# report_var_dict("Y (stations)", Y)
# report_var_dict("X (flows)", X)
# report_var_dict("S (secondary)", S)
# report_var_dict("G (grid)", G)
# report_var_dict("Z (aux)", Z)

# # If you suspect key-ordering issues, show the first few keys for each dict:
# print("\nExample keys (first 10) per variable dict to check ordering:")
# for name, d in [("X", X), ("S", S), ("G", G), ("Y", Y), ("Z", Z)]:
#     try:
#         keys = list(d.keys())[:10]
#     except Exception:
#         keys = "cannot list keys (not a dict)"
#     print(f"  {name}: {keys}")

# # Efficient vectorized read (useful for very large dicts)
# def sum_var_values(vdict):
#     varlist = list(vdict.values())
#     if not varlist:
#         return 0.0
#     vals = md.getAttr('X', varlist)
#     return sum(abs(v) for v in vals), len(varlist)

# for name,d in [("X", X), ("G", G), ("S", S), ("Y", Y), ("Z", Z), ("INV", INV)]:
#     s, nvars = sum_var_values(d)
#     print(f"{name}: sum abs values = {s:.6f} over {nvars} vars")

# # Check whether LC and TT are Vars or expressions and print values
# for key in ["LC","TT"]:
#     item = F.get(key, None)
#     if item is None:
#         print(f"F['{key}'] not present")
#     else:
#         if isinstance(item, gp.Var):
#             print(f"F['{key}'] is a Var with .X = {item.X}")
#         else:
#             # compute its numeric value from components if possible
#             try:
#                 # If you have stored cost terms or can build a quick expression, compute it explicitly.
#                 print(f"F['{key}'] is not a Var (type {type(item)}). You must compute its value manually.")
#             except Exception as e:
#                 print(f"  Couldn't evaluate F['{key}']: {e}")

# print("\nDiagnostic block finished. If any dict shows 'all zero' but total nonzero vars > 0, it means your filtering logic removed them earlier.")
# # -----------------------------------------------------------------